# Inflation Nowcasting — Replication Skeleton
**Knotek & Zaman (2014), Cleveland Fed WP 14-03** · companion to `ROADMAP.md` (the plan) and `paper_reference.md` (the ground truth).

You implement every `TODO`. Self-test cells are provided — green means move on.
Solutions live in `solutions/` — rules for opening them are in ROADMAP.md.
Requires internet on the first run: official FRED API if `FRED_API_KEY` is set (see the M0 walkthrough below), else a no-key fallback endpoint. Series are cached in `data/` so later runs work offline. Python: pandas, numpy, statsmodels, matplotlib (+ scikit-learn for M8).

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (10, 3.5)

## M0 — Data (provided)

### How the data download works — FRED API walkthrough

**FRED** (Federal Reserve Economic Data, St. Louis Fed) has an official REST API: every request is just a URL like
`https://api.stlouisfed.org/fred/series/observations?series_id=CPIAUCSL&api_key=YOUR_KEY&file_type=json`
— authenticated by a free API key, rate-limited at ~120 requests/minute (we make 8–9). The **`fredapi`** package is a thin Python wrapper around it: `Fred(api_key=...).get_series("CPIAUCSL")` returns a pandas Series. It's already installed in the anaconda3 environment.

Why the official API instead of the old `fredgraph.csv` no-key trick: it's documented and stable, exposes series metadata, and — crucial for **M7** — serves ALFRED *real-time vintages* via `Fred(...).get_series_as_of_date(code, date)` (what the data looked like on a given day, before revisions).

**One-time setup** (the key lives in your shell, never in this notebook — this folder is shared with AI tutor sessions and pushed to GitHub):

1. Get a key at <https://fred.stlouisfed.org/docs/api/api_key.html> (done).
2. In a terminal: `echo 'export FRED_API_KEY="YOUR_KEY"' >> ~/.zshrc`, then open a **fresh** terminal and restart Jupyter from it — Jupyter only sees env vars that existed when it launched.
3. Sanity check in a cell: `import os; print(os.environ.get("FRED_API_KEY") is not None)` → `True`.

**Caching:** each series is saved to `data/<CODE>.csv` on first download, so re-running the notebook is instant and works offline. The trade-off: the cache freezes the data at download time. After a new CPI/PCE release — or for genuinely *live* nowcasting — re-fetch with `fetch_fred(code, refresh=True)`, or just delete the `data/` folder. (`data/` is git-ignored.)

In [ ]:
# --- M0 (provided): data download -------------------------------------------
# Preferred path: official FRED API via `fredapi`, key in the FRED_API_KEY env
# var (setup: see the walkthrough cell above). Falls back to FRED's unofficial
# no-key CSV endpoint if the key is missing. Series are cached in data/ so
# reruns work offline; pass refresh=True after new data releases.

import os
from pathlib import Path

FRED_API_KEY = os.environ.get("FRED_API_KEY")
CACHE_DIR = Path("data")
CACHE_DIR.mkdir(exist_ok=True)


def fetch_fred(code: str, start: str = "1993-01-01", refresh: bool = False) -> pd.Series:
    cache = CACHE_DIR / f"{code}.csv"
    if cache.exists() and not refresh:
        s = pd.read_csv(cache, index_col=0, parse_dates=True).iloc[:, 0]
    else:
        if FRED_API_KEY:
            from fredapi import Fred
            s = Fred(api_key=FRED_API_KEY).get_series(code, observation_start=start)
        else:
            url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={code}"
            s = pd.read_csv(url, index_col=0, parse_dates=True).iloc[:, 0]
        s.to_frame(code).to_csv(cache)
    s = pd.to_numeric(s, errors="coerce").dropna()
    s.name = code
    return s[s.index >= start]

CODES = {
    "cpi":      "CPIAUCSL",        # CPI all items, SA, monthly
    "core_cpi": "CPILFESL",        # CPI ex food & energy, SA, monthly
    "food_cpi": "CPIUFDSL",        # CPI food, SA, monthly
    "gas_cpi":  "CUSR0000SETB01",  # CPI gasoline (all types), SA, monthly
    "pce":      "PCEPI",           # PCE price index, monthly
    "core_pce": "PCEPILFE",        # core PCE price index, monthly
    "gas_wk":   "GASALLW",         # EIA retail gasoline, all grades, NSA, weekly
    "oil":      "DCOILBRENTEU",    # Brent crude spot, daily
}

LVL = {k: fetch_fred(v) for k, v in CODES.items()}
print("source:", "official FRED API" if FRED_API_KEY else
      "no-key CSV fallback (set FRED_API_KEY to use the official API)")
for k, s in LVL.items():
    print(f"{k:9s} {s.index[0].date()} -> {s.index[-1].date()}  n={len(s)}")

In [ ]:
# ### Self-test M0 (provided — do not edit) ###
assert len(LVL) == 8
for k in ["cpi", "core_cpi", "food_cpi", "gas_cpi", "pce", "core_pce"]:
    assert LVL[k].index.freqstr or (LVL[k].index.to_series().diff().dt.days.dropna().between(28, 31).mean() > 0.95), k
assert LVL["gas_wk"].index.to_series().diff().dt.days.dropna().mode()[0] == 7
assert LVL["oil"].index.to_series().diff().dt.days.dropna().mode()[0] == 1
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for ax, (k, s) in zip(axes.flat, LVL.items()):
    s.plot(ax=ax, title=k)
plt.tight_layout(); plt.show()
print("M0 OK — 6 monthly, 1 weekly, 1 daily series")

### ✍️ Checkpoint M0 — answer in writing before continuing

Why does the paper use SA data everywhere except weekly retail gasoline? What breaks if you mix SA and NSA in the same regression?

**My answers:**

- 


## M1 — Inflation arithmetic (paper eqs 1–2)

In [ ]:
def monthly_inflation(levels: pd.Series) -> pd.Series:
    """Month-over-month inflation in PERCENT (not annualized).
    Input: monthly price levels. Output: 100*(P_t/P_{t-1} - 1)."""
    # TODO(M1): ~1 line
    raise NotImplementedError


def quarterly_annualized(levels: pd.Series) -> pd.Series:
    """Quarterly annualized inflation per paper eqs (1)-(2).
    Step 1: quarterly level = MEAN of the 3 monthly levels
            (hint: levels.groupby(levels.index.to_period("Q")).mean()).
    Step 2: 100 * ((Q_T / Q_{T-1})**4 - 1)."""
    # TODO(M1): ~3 lines
    raise NotImplementedError

In [ ]:
# ### Self-test M1 (provided — do not edit) ###
# Cross-check against BEA's own quarterly PCE price index (independent series).
pcectpi = fetch_fred("PCECTPI")
bea_q = (100 * ((pcectpi / pcectpi.shift(1)) ** 4 - 1)).dropna()
bea_q.index = bea_q.index.to_period("Q")
mine = quarterly_annualized(LVL["pce"]).dropna()
common = mine.index.intersection(bea_q.index)[-20:]
gap = (mine[common] - bea_q[common]).abs()
print(gap.describe())
assert gap.max() < 0.2, "quarterly aggregation does not match BEA convention"
print("M1 OK — your eq(1)-(2) matches BEA's published quarterly PCE inflation")

### ✍️ Checkpoint M1 — answer in writing before continuing

1. Why does averaging three monthly *levels* differ from averaging three monthly *rates*?
2. Which months of quarter T−1 still influence quarter T inflation, and why does that give the model a head start before T begins?

**My answers:**

- 


## M2 — Core & food default: recursive MA12 (paper eq 4)

In [ ]:
def ma12_forecast(infl: pd.Series, n_ahead: int = 1) -> np.ndarray:
    """Paper eq (4): RECURSIVE 12-month moving average forecast.
    'Recursive' = each forecast is appended to the history and used in the
    next step's 12-month average (NOT a frozen trailing window).
    Input: realized monthly inflation (dropna'd). Output: n_ahead forecasts."""
    # TODO(M2): ~6 lines (a list, a loop, np.mean of the last 12)
    raise NotImplementedError

In [ ]:
# ### Self-test M2 (provided — do not edit) ###
const = pd.Series(0.2, index=pd.date_range("2000-01-01", periods=24, freq="MS"))
assert np.allclose(ma12_forecast(const, 6), 0.2), "constant series must forecast itself"

core = monthly_inflation(LVL["core_cpi"]).dropna()
eval_idx = core.loc["2005":"2019"].index
err_ma, err_rw = [], []
for t in eval_idx:
    hist = core.loc[:t].iloc[:-1]          # info: data through t-1 only
    err_ma.append(core[t] - ma12_forecast(hist, 1)[0])
    err_rw.append(core[t] - hist.iloc[-1])
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
print(f"RMSE  MA12: {rmse(err_ma):.4f}   random walk: {rmse(err_rw):.4f}")
assert rmse(err_ma) < rmse(err_rw), "MA12 should beat the monthly random walk (Atkeson-Ohanian logic)"
print("M2 OK")

### ✍️ Checkpoint M2 — answer in writing before continuing

Why is this naive rule so hard to beat for core inflation at short horizons? What time-series property of core inflation does it exploit?

**My answers:**

- 


## M3 — The gasoline block (paper eqs 6–7 + footnote 7) — hardest milestone

In [ ]:
def weekly_to_monthly(weekly: pd.Series) -> pd.Series:
    """Average weekly observations within each calendar month.
    Return a monthly series indexed by month-START timestamps (to align with FRED)."""
    # TODO(M3a): ~3 lines (groupby to_period("M"), mean, to_timestamp)
    raise NotImplementedError


def seasonally_adjust_gas(nsa_infl: pd.Series, cpi_gas_infl: pd.Series) -> pd.Series:
    """Paper footnote 7. For each month, the seasonal factor is the average of
    (NSA gasoline inflation - SA CPI-gasoline inflation) in the SAME calendar
    month over the PREVIOUS 3 years. Return nsa_infl - sf.
    Careful: no look-ahead — only past years may enter the factor.
    Hint: diff.groupby(diff.index.month).transform(lambda x: x.shift(1).rolling(3).mean())"""
    # TODO(M3b): ~4 lines
    raise NotImplementedError


def gas_forecast_from_oil(gas_m: pd.Series, oil_m: pd.Series, oil_hat: float,
                          window: int = 60) -> float:
    """Paper eqs (6)-(7). Two-stage regression on the last `window` months of LEVELS:
    (6) long-run:  gas = alpha + beta*oil            -> fitted gap
    (7) ECM:       d(gas)_t = b*d(oil)_t + c*gap_{t-1}
    Then forecast next month's gas level using oil_hat (random-walk oil forecast).
    Returns the forecasted gasoline price LEVEL for the next month."""
    # TODO(M3c): ~10 lines with statsmodels OLS
    raise NotImplementedError

In [ ]:
# ### Self-test M3 (provided — do not edit) ###
gas_m = weekly_to_monthly(LVL["gas_wk"])                    # $ / gallon, monthly avg
gas_nsa_infl = monthly_inflation(gas_m)
cpi_gas_infl = monthly_inflation(LVL["gas_cpi"])
gas_sa = seasonally_adjust_gas(gas_nsa_infl, cpi_gas_infl)

common = gas_sa.loc["2000":"2019"].index.intersection(cpi_gas_infl.index)
corr = gas_sa[common].corr(cpi_gas_infl[common])
print(f"corr(SA EIA gasoline inflation, CPI gasoline inflation) = {corr:.3f}")
assert corr > 0.9, "EIA-based SA gasoline inflation should track CPI gasoline closely"

oil_m = weekly_to_monthly(LVL["oil"])                       # daily -> monthly avg
oil_hat = float(LVL["oil"].iloc[-1])                        # random-walk oil forecast
lr_beta = sm.OLS(*[x.iloc[-60:] for x in
                   [pd.concat({"g": gas_m, "o": oil_m}, axis=1).dropna()["g"],
                    sm.add_constant(pd.concat({"g": gas_m, "o": oil_m}, axis=1).dropna()["o"])]]).fit().params.iloc[1]
assert lr_beta > 0, "long-run oil coefficient must be positive"
fcst = gas_forecast_from_oil(gas_m, oil_m, oil_hat)
print(f"next-month gasoline level forecast: {fcst:.2f} (last actual {gas_m.iloc[-1]:.2f})")
assert 0.5 * gas_m.iloc[-1] < fcst < 1.5 * gas_m.iloc[-1], "forecast should be near the last level"
print("M3 OK")

### ✍️ Checkpoint M3 — answer in writing before continuing

1. Why seasonally adjust with CPI-gasoline seasonal factors rather than running X-13 on the EIA series?
2. What pump-price behavior does the ECM's gap term capture?
3. Why does the paper use a random walk for oil instead of futures prices (see its footnote 9)?

**My answers:**

- 


## M4 — Bridge equations (paper eqs 5 & 9)

In [ ]:
def bridge_forecast(y_infl: pd.Series, x_infl: pd.Series, window: int = 24) -> float:
    """Paper eqs (5)/(9): bridge from a released series to an unreleased one.
    Fit OLS  y_t = b + a*x_t  on the last `window` months where BOTH are observed,
    then predict y for the newest month where ONLY x is observed.
    Returns that single predicted value."""
    # TODO(M4): ~5 lines. Careful: same-month pairing, not lagged.
    raise NotImplementedError

In [ ]:
# ### Self-test M4 (provided — do not edit) ###
core_cpi_i = monthly_inflation(LVL["core_cpi"]).dropna()
core_pce_i = monthly_inflation(LVL["core_pce"]).dropna()
eval_idx = core_pce_i.loc["2005":"2019"].index
err_br, err_ma = [], []
for t in eval_idx:
    y_hist = core_pce_i.loc[:t].iloc[:-1]         # PCE not yet out for month t
    x_hist = core_cpi_i.loc[:t]                   # CPI IS out for month t
    err_br.append(core_pce_i[t] - bridge_forecast(y_hist, x_hist))
    err_ma.append(core_pce_i[t] - ma12_forecast(y_hist, 1)[0])
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
print(f"RMSE  bridge: {rmse(err_br):.4f}   MA12: {rmse(err_ma):.4f}")
assert rmse(err_br) < rmse(err_ma), "bridging released core CPI should beat MA12 for core PCE (paper Table 7, line 14)"
print("M4 OK")

### ✍️ Checkpoint M4 — answer in writing before continuing

1. What do CPI and PCE share that makes a same-month bridge work? Name two reasons they diverge.
2. Why a 24-month window instead of the full sample? (Connect to the bias-variance / regime-change argument.)

**My answers:**

- 


## M5 — Headline regression + deterministic model switching (paper eqs 8–11)

In [ ]:
def choose_regime(have_own_cpi_t: bool, have_gas_t: bool) -> str:
    """Deterministic model switching (the paper's core innovation, eqs 9-11).
    Return one of: "bridge" (eq 9), "disagg_reg" (eq 10), "ma" (eq 11)."""
    # TODO(M5a): 3 lines. Think about precedence: a released CPI beats everything.
    raise NotImplementedError


def headline_reg_forecast(y_infl: pd.Series, core_infl: pd.Series,
                          food_infl: pd.Series, gas_infl: pd.Series,
                          core_hat: float, food_hat: float, gas_hat: float,
                          window: int = 24) -> float:
    """Paper eq (10) for ONE headline measure. Note the zero restrictions:
    the CPI equation uses core CPI; the PCE equation uses core PCE; both use
    food and gasoline. Fit OLS  y = b + c1*core + c3*food + c4*gas  on the last
    `window` months where all four are observed; predict with the hats."""
    # TODO(M5b): ~7 lines
    raise NotImplementedError

In [ ]:
# ### Self-test M5 (provided — do not edit) ###
assert choose_regime(True,  True)  == "bridge"
assert choose_regime(True,  False) == "bridge"
assert choose_regime(False, True)  == "disagg_reg"
assert choose_regime(False, False) == "ma"

cpi_i  = monthly_inflation(LVL["cpi"]).dropna()
food_i = monthly_inflation(LVL["food_cpi"]).dropna()
df = pd.concat({"y": cpi_i, "core": core_cpi_i, "food": food_i, "gas": gas_sa}, axis=1).dropna()
r2 = sm.OLS(df["y"], sm.add_constant(df[["core", "food", "gas"]])).fit().rsquared
print(f"full-sample R2 of eq(10) CPI regression: {r2:.3f}")
assert r2 > 0.5, "core+food+gas should explain most headline CPI variation"
print("M5 OK")

### ✍️ Checkpoint M5 — answer in writing before continuing

Write the regime table from memory: for information states (i) nothing released, no gas; (ii) weekly gas only; (iii) CPI(t) released — which equation governs, and why is this 'time-varying weights' in the eq (3) sense?

**My answers:**

- 


## M6 — Calendar simulation

We evaluate the model at three information dates ("cases") for each target month *t* — a simplified version of the paper's six (Table 2):

| Case | Stand-in date            | CPI thru | PCE thru | EIA gas in month t | Oil        |
|------|--------------------------|----------|----------|--------------------|------------|
| A    | last day of month t−1    | t−2      | t−2      | none               | thru t−1   |
| B    | ~15th of month t         | t−1      | t−2      | first 2 weeks      | thru day 14|
| C    | day after CPI(t) release | t        | t−1      | all weeks          | all        |

Your job: implement `nowcast_month(t, case)` returning nowcasts for all four measures,
respecting ONLY the information in the table (this discipline is the whole exercise).
Then the provided loop computes RMSEs per case. Deviation from the paper (noted in
ROADMAP): we evaluate 2005–2019 on final-vintage data, so error LEVELS won't match the
paper's tables — the declining-RMSE PATTERN is the success criterion.

In [ ]:
# Realized monthly inflation series (info truncation happens inside nowcast_month)
INFL = {
    "cpi":      monthly_inflation(LVL["cpi"]).dropna(),
    "core_cpi": monthly_inflation(LVL["core_cpi"]).dropna(),
    "food":     monthly_inflation(LVL["food_cpi"]).dropna(),
    "pce":      monthly_inflation(LVL["pce"]).dropna(),
    "core_pce": monthly_inflation(LVL["core_pce"]).dropna(),
    "gas_sa":   gas_sa,
}
GAS_M, OIL_D = gas_m, LVL["oil"]


def nowcast_month(t: pd.Timestamp, case: str) -> dict:
    """Nowcast {'cpi','core_cpi','pce','core_pce'} for target month t under case A/B/C.

    Recipe (follow paper_reference.md §3):
      1. Truncate every series to what the case table allows. For case B's partial
         gasoline month: average EIA weeks with day <= 14 in month t.
      2. core_cpi & food: actual if released, else ma12_forecast (recursive, chained
         over however many months are missing).
      3. core_pce: actual if released; bridge_forecast from core_cpi if CPI is one
         month ahead of PCE; else ma12.
      4. gas: SA partial-month estimate if weeks exist (case B/C), else
         gas_forecast_from_oil with a random-walk oil forecast (case A);
         convert the level forecast to inflation, then seasonally adjust.
      5. headline: actual if released (case C cpi); else choose_regime ->
         headline_reg_forecast with the hats from steps 2-4; pce in case C: bridge.
    """
    # TODO(M6): the capstone. ~40 lines. Build it case by case; test case C first
    # (easiest: only PCE measures need nowcasting), then B, then A.
    raise NotImplementedError

In [ ]:
# ### Self-test M6 (provided — do not edit) ###
targets = INFL["pce"].loc["2005":"2019"].index
results, bias, skipped = {}, {}, {}
for case in ["A", "B", "C"]:
    errs = {k: [] for k in ["cpi", "core_cpi", "pce", "core_pce"]}
    n_skip = 0
    for t in targets:
        try:
            nc = nowcast_month(t, case)
        except Exception:
            n_skip += 1
            continue
        for k in errs:
            errs[k].append(nc[k] - INFL[k][t])
    skipped[case] = n_skip
    results[case] = {k: float(np.sqrt(np.mean(np.square(v)))) for k, v in errs.items()}
    bias[case] = {k: float(np.mean(v)) for k, v in errs.items()}
tbl = pd.DataFrame(results).T
print("RMSE by case:"); print(tbl.round(4))
print("\nMean error (bias) by case:"); print(pd.DataFrame(bias).T.round(4))
print(f"\nskipped months per case (of {len(targets)} targets): {skipped}")

tbl[["cpi", "core_cpi"]].plot(kind="bar", title="Monthly nowcast RMSE by case (paper Fig. 2 analogue)")
plt.ylabel("pp, month-over-month"); plt.show()

for case, n_skip in skipped.items():
    assert n_skip <= 0.1 * len(targets), \
        f"case {case}: nowcast_month raised on {n_skip}/{len(targets)} months — fix the errors, don't rely on skips"
assert tbl.loc["C", "cpi"] < tbl.loc["B", "cpi"] < tbl.loc["A", "cpi"], \
    "headline CPI RMSE must fall as information arrives"
assert (tbl.loc["A", "core_cpi"] - tbl.loc["B", "core_cpi"]) < \
       (tbl.loc["A", "cpi"] - tbl.loc["B", "cpi"]), \
    "core should improve far less than headline (gasoline is doing the work)"
print("M6 OK — you have replicated the paper's signature result")

### ✍️ Checkpoint M6 — answer in writing before continuing

Explain in one sentence why core RMSE barely moves from case A to C while headline RMSE falls sharply.

**My answers:**

- 


## M7 — Stretch goals (see ROADMAP.md)

Pick any: ALFRED real-time vintages (`fredapi.Fred.get_series_as_of_date` — the
`FRED_API_KEY` setup from M0 already unlocks this) · Diebold–Mariano tests ·
quarterly aggregation (reuse `quarterly_annualized` and the paper's 14 cases).

The ML extension has been promoted to a full milestone: **M8**, below.

## M8 — Interpretable ML horse-race (ROADMAP M8)

ML models enter only if their fitted behavior can be *read* — coefficients (M8a),
selection sets (M8b), shape functions (M8c). Black boxes appear only as accuracy
ceilings to interpret against. All sklearn; requires M5–M6 to be green (reuses
`headline_reg_forecast`, `INFL`, `gas_sa`).

**Discipline (non-negotiable, inherited from the paper):** any tuning uses data
*before* the forecast date only, and the benchmark to beat is **your eq-(10)
model**, not the random walk.

**The harness** (provided below) evaluates case-B style with *perfect
disaggregates*: every model sees regressor histories through t−1 and the actual
month-t disaggregates as hats. Hat generation is identical across model classes,
so this isolates the question that M8 actually asks: *which regression maps
disaggregates to headline best?*

In [ ]:
# --- M8 (provided): extra data + evaluation harness --------------------------
ML_CODES = {
    "ppi_gas":  "WPS0571",      # PPI gasoline, SA, monthly
    "imports":  "IR",           # import price index, all commodities, NSA, monthly
    "wages":    "AHETPI",       # avg hourly earnings, production workers, SA, monthly
    "gas_spot": "DGASUSGULF",   # Gulf Coast conventional gasoline spot, daily
}
ML_LVL = {k: fetch_fred(v) for k, v in ML_CODES.items()}
ML_INFL = {k: monthly_inflation(weekly_to_monthly(s) if k == "gas_spot" else s).dropna()
           for k, s in ML_LVL.items()}

HARNESS_IDX = INFL["cpi"].loc["2005":"2019"].index
print("harness:", len(HARNESS_IDX), "target months, case-B info, perfect disaggregates")

In [ ]:
def ridge_headline_forecast(y_infl: pd.Series, core_infl: pd.Series,
                            food_infl: pd.Series, gas_infl: pd.Series,
                            core_hat: float, food_hat: float, gas_hat: float,
                            alpha: float = 1.0, window: int = 24) -> float:
    """M8a: paper eq (10) with an L2 penalty (ridge).
    Same recipe as headline_reg_forecast, but:
      - standardize the three regressors with the WINDOW's mean/std only
        (sklearn StandardScaler; scaling on the full sample is leakage),
      - fit sklearn Ridge(alpha=alpha); fit_intercept=True keeps the
        intercept unpenalized (never penalize the intercept),
      - predict with the standardized hats.
    alpha=0 must reproduce the OLS forecast; alpha -> inf collapses to the
    window mean of y."""
    # TODO(M8a): ~8 lines (StandardScaler + Ridge)
    raise NotImplementedError

In [ ]:
# ### Self-test M8a (provided — do not edit) ###
from sklearn.linear_model import Ridge as _Ridge

# 1) limits: alpha=0 == OLS; alpha -> inf == window mean of y
args = (INFL["cpi"], INFL["core_cpi"], INFL["food"], INFL["gas_sa"], 0.2, 0.15, 1.0)
f_ols = headline_reg_forecast(*args)
assert abs(ridge_headline_forecast(*args, alpha=0.0) - f_ols) < 1e-6, "alpha=0 must equal OLS"
wmean = pd.concat({"y": INFL["cpi"], "core": INFL["core_cpi"], "food": INFL["food"],
                   "gas": INFL["gas_sa"]}, axis=1).dropna().iloc[-24:]["y"].mean()
assert abs(ridge_headline_forecast(*args, alpha=1e9) - wmean) < 1e-3, "alpha->inf must collapse to window mean"

# 2) expanding-window-tuned ridge vs eq(10) OLS on the harness (tuning uses PAST errors only)
grid = [0.0, 0.3, 1.0, 3.0, 10.0]
sq = {a: [] for a in grid}
err_tuned, err_ols = [], []
for t in HARNESS_IDX:
    hists = (INFL["cpi"].loc[:t].iloc[:-1], INFL["core_cpi"].loc[:t].iloc[:-1],
             INFL["food"].loc[:t].iloc[:-1], INFL["gas_sa"].loc[:t].iloc[:-1])
    hats = (INFL["core_cpi"][t], INFL["food"][t], INFL["gas_sa"][t])
    preds = {a: ridge_headline_forecast(*hists, *hats, alpha=a) for a in grid}
    if len(sq[0.0]) >= 24:
        a_star = min(grid, key=lambda a: np.mean(sq[a][-24:]))
        err_tuned.append(preds[a_star] - INFL["cpi"][t])
        err_ols.append(preds[0.0] - INFL["cpi"][t])
    for a in grid:
        sq[a].append((preds[a] - INFL["cpi"][t]) ** 2)
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
print(f"harness RMSE  ridge(tuned): {rmse(err_tuned):.4f}   eq(10) OLS: {rmse(err_ols):.4f}")
assert rmse(err_tuned) <= 1.05 * rmse(err_ols), "tuned ridge should not lose to OLS by more than 5%"

# 3) rolling coefficient paths (standardized, so comparable): the instability story
dfp = pd.concat({"y": INFL["cpi"], "core": INFL["core_cpi"],
                 "food": INFL["food"], "gas": INFL["gas_sa"]}, axis=1).dropna()
rows = []
for i in range(24, len(dfp)):
    w = dfp.iloc[i - 24:i]
    Xs = (w[["core", "food", "gas"]] - w[["core", "food", "gas"]].mean()) / w[["core", "food", "gas"]].std()
    ols_g = sm.OLS(w["y"], sm.add_constant(Xs)).fit().params["gas"]
    rdg_g = _Ridge(alpha=10.0).fit(Xs, w["y"]).coef_[2]
    rows.append({"date": dfp.index[i], "OLS": ols_g, "ridge(alpha=10)": rdg_g})
pd.DataFrame(rows).set_index("date").plot(
    title="Rolling 24m gasoline coefficient (standardized): OLS jumps, ridge smooths")
plt.ylabel("coef on gas"); plt.show()
print("M8a OK")

### ✍️ Checkpoint M8a — answer in writing before continuing

1. Why do short rolling windows and shrinkage address the same problem? What is that problem, in this paper's setting?
2. Under what data conditions would ridge on a 60-month window beat OLS on a 24-month window?

**My answers:**

- 


In [ ]:
def lasso_selection(y_infl: pd.Series, X: pd.DataFrame, window: int = 60,
                    step: int = 3) -> pd.DataFrame:
    """M8b: rolling LASSO variable selection.
    For each rolling window of `window` consecutive months (advancing by
    `step` months):
      - standardize X inside the window (full-sample scaling is leakage),
      - fit sklearn LassoCV(cv=TimeSeriesSplit(n_splits=4)) on (X, y),
      - record which columns get nonzero coefficients.
    Return a DataFrame: index = window-end date, columns = X.columns,
    values = bool (selected in that window)."""
    # TODO(M8b): ~10 lines (LassoCV, TimeSeriesSplit, StandardScaler)
    raise NotImplementedError

In [ ]:
# ### Self-test M8b (provided — do not edit) ###
X_wide = pd.concat({
    "core_cpi": INFL["core_cpi"], "food": INFL["food"], "gas_sa": INFL["gas_sa"],
    "ppi_gas": ML_INFL["ppi_gas"], "imports": ML_INFL["imports"],
    "wages": ML_INFL["wages"], "gas_spot": ML_INFL["gas_spot"],
}, axis=1).dropna().loc["2000":"2019"]
y_wide = INFL["cpi"][X_wide.index]

sel = lasso_selection(y_wide, X_wide)
assert list(sel.columns) == list(X_wide.columns) and sel.dtypes.eq(bool).all()
freq = sel.mean().sort_values(ascending=False)
print("selection frequency across windows:")
print(freq.round(2))
gas_family = sel[["gas_sa", "ppi_gas", "gas_spot"]].any(axis=1).mean()
print(f"\ngasoline family (any of gas_sa/ppi_gas/gas_spot) selected in {gas_family:.0%} of windows")
assert gas_family >= 0.8, "a gasoline-family regressor should be selected in >=80% of windows"
print("M8b OK")

### ✍️ Checkpoint M8b — answer in writing before continuing

1. Which regressors survive, and how stable is the selection across windows? Did LASSO rediscover the paper's hand-picked four?
2. The gasoline "twins" (CPI gas, PPI gas, spot gas) are nearly collinear. What does LASSO do with collinear twins, and why does that mean you should report the selection *family*, not the individual pick?

**My answers:**

- 


In [ ]:
def spline_additive_forecast(y_infl: pd.Series, X_hist: pd.DataFrame, x_hat: dict,
                             window: int = 60, n_knots: int = 5,
                             alpha: float = 1.0) -> float:
    """M8c: DIY GAM. One sklearn Pipeline:
    StandardScaler -> SplineTransformer(n_knots=n_knots, degree=3) -> Ridge(alpha).
    SplineTransformer expands EACH feature into its own spline basis, and Ridge
    is linear in that basis, so the model is ADDITIVE:
        y = f_core(core) + f_food(food) + f_gas(gas)
    — nonlinear per feature, but each fitted shape f_j can be plotted and read.
    Fit on the last `window` months where y and all X_hist columns are observed;
    return the prediction at x_hat (a dict keyed by X_hist's columns).
    Note window=60, not 24: splines need more data — a deliberate, disclosed
    deviation from eq (10)'s window."""
    # TODO(M8c): ~7 lines (make_pipeline, fit, predict)
    raise NotImplementedError

In [ ]:
# ### Self-test M8c (provided — do not edit) ###
from sklearn.ensemble import HistGradientBoostingRegressor

X3 = pd.concat({"core": INFL["core_cpi"], "food": INFL["food"],
                "gas": INFL["gas_sa"]}, axis=1).dropna()
y3 = INFL["cpi"]

err = {"spline (DIY GAM)": [], "eq(10) OLS": [], "boosting ceiling": [], "random walk": []}
for t in HARNESS_IDX:
    Xh, yh = X3.loc[:t].iloc[:-1], y3.loc[:t].iloc[:-1]
    hats = {"core": INFL["core_cpi"][t], "food": INFL["food"][t], "gas": INFL["gas_sa"][t]}
    err["spline (DIY GAM)"].append(spline_additive_forecast(yh, Xh, hats) - y3[t])
    err["eq(10) OLS"].append(headline_reg_forecast(yh, Xh["core"], Xh["food"], Xh["gas"],
                                                   hats["core"], hats["food"], hats["gas"]) - y3[t])
    w = pd.concat([yh.rename("y"), Xh], axis=1).dropna().iloc[-60:]
    hgb = HistGradientBoostingRegressor(max_iter=100, random_state=0).fit(
        w[["core", "food", "gas"]], w["y"])
    err["boosting ceiling"].append(float(hgb.predict(pd.DataFrame([hats]))[0]) - y3[t])
    err["random walk"].append(float(yh.iloc[-1]) - y3[t])
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
res = pd.Series({k: rmse(v) for k, v in err.items()}).sort_values()
print("harness RMSE (headline CPI, perfect disaggregates):")
print(res.round(4))
assert res["spline (DIY GAM)"] < res["random walk"], "the additive spline model must beat the monthly random walk"

# gasoline shape function: sweep gas over its range, others held at window medians
w = pd.concat([y3.rename("y"), X3], axis=1).dropna().iloc[-60:]
gas_grid = np.linspace(w["gas"].quantile(0.02), w["gas"].quantile(0.98), 60)
shape = [spline_additive_forecast(w["y"], w[["core", "food", "gas"]],
                                  {"core": w["core"].median(), "food": w["food"].median(),
                                   "gas": g}) for g in gas_grid]
plt.plot(gas_grid, shape); plt.axvline(0, ls=":", c="gray")
plt.xlabel("gasoline inflation (pp, m/m)"); plt.ylabel("predicted headline CPI inflation")
plt.title("DIY-GAM gasoline shape function — look for asymmetry around 0")
plt.show()
print("M8c OK")

### ✍️ Checkpoint M8c — answer in writing before continuing

1. From the fitted gasoline shape function: is the pass-through asymmetric ("rockets and feathers")? How would you tell noise from signal here?
2. Why does additivity keep this model readable while a random forest is not?
3. Did nonlinearity actually pay in RMSE? If not, why might 60 monthly observations be too few for it to?

### ✍️ Closing checkpoint M8

Which of the paper's design choices did you keep in the ML version, and why? (Think: calendar discipline, information sets, past-only tuning, benchmark choice.)

**My answers:**

- 
